# Concours MTH3302

## Prédiction de la consommation en carburant de voitures récentes.

### Contexte
Une gestion efficace de la consommation de carburant devient un enjeu crucial tant pour les conducteurs que pour l’industrie automobile, particulièrement dans le contexte actuel de transition énergétique et de réduction des émissions de gaz à effet de serre. La consommation en carburant des véhicules récents dépend de plusieurs caractéristiques techniques telles que la boîte de vitesses, la cylindrée, le nombre de cylindres et le type de transmission. Ces variables influencent directement l'efficacité énergétique et peuvent varier d’un véhicule à l’autre.

### Objectif

Dans cette étude, nous nous concentrons sur la prédiction de la consommation en carburant de voitures récentes. À partir d’un jeu de données comprenant la consommation moyenne en litres pour 100 kilomètres (L/100km) de près de 400 véhicules, ainsi que leurs caractéristiques techniques, l’objectif est de prédire la consommation en carburant pour un ensemble de test en fonction de ces différentes caractéristiques. Ce modèle prédictif permettra d’évaluer plus précisément les performances de consommation des véhicules et d'aider à identifier les facteurs déterminants pour l’optimisation de la consommation de carburant.

### Variables

La variable d'intérêt est la **consommation** en L/100km.

Les variables explicatives sont les suivantes:
- année: l'année du modèle
- type: le type de véhicule
- nombre_cylindres: le nombre de cylindres du moteur
- cylindree: la cylindrée du moteur en L
- transmission: le type de transmission (propulsion, traction, 4x4 et intégrale)
- boite: le type de boite de vitesses (automatique ou manuelle)



In [ ]:
using CSV
using GLM
using MLJ
using DataFrames
using Gadfly
using Random
using Statistics
using Combinatorics
using LinearAlgebra
using HypothesisTests
# using StatsPlots
using StatsModels
using CategoricalArrays
using StatsBase

In [ ]:
Random.seed!(4355)

In [ ]:
train_data = CSV.read("train.csv", DataFrame, decimal=',')

train = deepcopy(train_data)
first(train_data, 5)

In [ ]:
MLJ.schema(train_data)

In [ ]:
describe(train_data)

In [ ]:
# analyser la consommation par cylindree
average_per_cylindree = combine(groupby(train, :cylindree), :consommation => mean)
plot(
    layer(train, x=:cylindree, y=:consommation, Geom.point),
    layer(average_per_cylindree, x=:cylindree, y=:consommation_mean, Geom.line),
    Guide.xlabel("cylindree"), Guide.ylabel("consommation")
)

In [ ]:
# analyser la consommation par cylindree
pp = [plot(train_data, x=:nombre_cylindres, y=:cylindree, Geom.point),
      plot(train_data, x=:cylindree, y=:nombre_cylindres, Geom.point)]
p = reshape(pp, (1,2))
set_default_plot_size(30cm, 10cm)
gridstack(p)

In [ ]:
# Afficher des diagrammes en boîte de consommation pour les variables du jeu de données
function box_plot_categorical(data)
    Gadfly.set_default_plot_size(30cm, 35cm)
    p1 = plot(data, x=:annee, y=:consommation, Geom.boxplot, Guide.title("Consommation par Année"), Guide.xlabel("Année"), Guide.ylabel("Consommation"))
    p2 = plot(data, x=:type, y=:consommation, Geom.boxplot, Guide.title("Consommation par Type de voiture"), Guide.xlabel("Type de voiture"), Guide.ylabel("Consommation"))
    p3 = plot(data, x=:nombre_cylindres, y=:consommation, Geom.boxplot, Guide.title("Consommation par nombre de cylindre"), Guide.xlabel("Nombre de cylindre"), Guide.ylabel("Consommation"))
    p4 = plot(data, x=:transmission, y=:consommation, Geom.boxplot, Guide.title("Consommation par type de transmission"), Guide.xlabel("Type de transmission"), Guide.ylabel("Consommation"))
    p5 = plot(data, x=:cylindree, y=:consommation, Geom.point, Geom.smooth(method=:lm), Guide.title("Nuage de points avec la droite de régression"), Guide.xlabel("Cylindree"), Guide.ylabel("Consommation"))
    p6 = plot(data, x=:boite, y=:consommation, Geom.boxplot, Guide.title("Consommation par type de boite de vitesse"), Guide.xlabel("Boite de vitesse"), Guide.ylabel("Consommation"))

    grid = vstack(hstack(p1, p2), hstack(p3, p4), hstack(p5, p6))
    display(grid)

    # réinitialiser la taille pour ne pas affecter les autres graphiques
    Gadfly.set_default_plot_size(20cm, 15cm)
end

In [ ]:
box_plot_categorical(train_data)

In [ ]:
train = CSV.read("train.csv", DataFrame, decimal=',')
test = CSV.read("test.csv", DataFrame, decimal=',');

# Partie 1
## Régressions linéaires simples

In [ ]:
y = train.consommation
n = length(y)

In [ ]:
function compute_residuals(model, y)
    ŷ = StatsModels.predict(model)
    res = (y - ŷ) / std(ŷ)

    return res
end

function plot_explanatory_variable(model, data, xlabel)
    predictions = StatsModels.predict(model)

    Gadfly.plot(
        x = data.x, 
        y = data.y,
        layer(
            x = data.x,
            y = predictions,
            Geom.line,
            Theme(default_color="red"),
        ),
        Geom.point,
        Guide.xlabel(xlabel), 
        Guide.ylabel("Consommation d'essence (L/100km)", orientation=:vertical),
    )
end

# function qqplot_test(model, data)
#     errors = compute_residuals(model, data.y)

#     qqnorm(errors, qqline = :R)
# end

function shapiro_wilk_test(model, data)
    errors = compute_residuals(model, data.y)
    p = pvalue(ShapiroWilkTest(errors))

    if p > 0.05
        println("$p > 0.05 -> On accepte l'hypothèse que les données proviennent d'une distribution normale")
    else 
        println("$p ≤ 0.05 -> On rejette l'hypothèse que les données proviennent d'une distribution normale")
    end
end

function residuals_vs_fitted_values_plot_test(model, data)
    errors = compute_residuals(model, data.y)
    fitted_values = fitted(model)

    Gadfly.plot(
        layer(x=fitted_values, y=errors, Geom.point),
        Guide.xlabel("Valeurs prédites"),
        Guide.ylabel("Résidus"),
    )
end

function residuals_vs_observation_order_plot_test(model, data)
    errors = compute_residuals(model, data.y)

    Gadfly.plot(
        layer(x=1:length(errors), y=errors, Geom.point),
        Guide.xlabel("Index"),
        Guide.ylabel("Résidus"),
    )
end

function get_data_set(seed, features, preprocess::Function = data -> nothing)
    Random.seed!(seed)
    data = CSV.read("train.csv", DataFrame, decimal=',')

    preprocess(data)

    train_id = sample(1:nrow(data), round(Int, .8*nrow(data)), ordered=true, replace=false)
    valid_id = setdiff(1:nrow(data), train_id)

    train = data[train_id,:]
    train = remove_outliers(train, features, :consommation)
    valid = data[valid_id,:]

    return train, valid
end

function align_categorical_features(train, valid, features)
    data = vcat(train, valid)

    categorical_features = filter(x -> eltype(data[:, x]) <: AbstractString || eltype(data[:, x]) <: CategoricalValue, features)
    for feature in categorical_features
        train_values = unique(train[!, feature])
        valid_values = unique(valid[!, feature])

        train_indices_to_remove = findall(x -> !(x in valid_values), train[!, feature])
        train = train[setdiff(1:nrow(train), train_indices_to_remove), :]

        valid_indices_to_remove = findall(x -> !(x in train_values), valid[!, feature])
        valid = valid[setdiff(1:nrow(valid), valid_indices_to_remove), :]
    end

    return train, valid
end

function update_features(data, features) 
    return vec(reduce(vcat, [
        filter(name -> startswith(name, string(feature)), names(data))
        for feature in features
    ]))
end

## 1.1 Analyse de la variable _nombre_cylindres_

In [ ]:
include("Jeremie_Utils.jl")

In [ ]:
x = string.(train.nombre_cylindres)
data = DataFrame(y = train.consommation, x = x)
model = create_model(data, [:x])

**Vérification de l'hypothèse de linéarité**

In [ ]:
plot_explanatory_variable(model, data, "Nombre de cylindres")

**Vérification de l'hypothèse de normalité des erreurs**

In [ ]:
# qqplot_test(model, data)

In [ ]:
shapiro_wilk_test(model, data)

**Vérification de l'hypothèse d'homosédasticité des erreurs**

In [ ]:
residuals_vs_fitted_values_plot_test(model, data)

**Vérification de l'hypothèse d'indépendance des erreurs**

In [ ]:
residuals_vs_observation_order_plot_test(model, data)

In [ ]:
r2(model)

## 1.2 Analyse de la variable _type_

In [ ]:
data = DataFrame(y = train.consommation, x = train.type)
model = create_model(data, [:x])

**Vérification de l'hypothèse de linéarité**

In [ ]:
plot_explanatory_variable(model, data, "Type")

**Vérification de l'hypothèse de normalité des erreurs**

In [ ]:
# qqplot_test(model, data)

In [ ]:
shapiro_wilk_test(model, data)

**Vérification de l'hypothèse d'homosédasticité des erreurs**

In [ ]:
residuals_vs_fitted_values_plot_test(model, data)

**Vérification de l'hypothèse d'indépendance des erreurs**

In [ ]:
residuals_vs_observation_order_plot_test(model, data)

In [ ]:
r2(model)

## 1.3 Analyse de la variable _cylindree_

In [ ]:
data = DataFrame(y = train.consommation, x = log.(train.cylindree))
model = create_model(data, [:x])

# data = DataFrame(y = train.consommation, x = train.cylindree, x² = train.cylindree .^ 2)
# model = create_model(data, [:x, :x²])

**Vérification de l'hypothèse de linéarité**

In [ ]:
plot_explanatory_variable(model, data, "Cylindrée")

**Vérification de l'hypothèse de normalité des erreurs**

In [ ]:
# qqplot_test(model, data)

In [ ]:
shapiro_wilk_test(model, data)

**Vérification de l'hypothèse d'homosédasticité des erreurs**

In [ ]:
residuals_vs_fitted_values_plot_test(model, data)

**Vérification de l'hypothèse d'indépendance des erreurs**

In [ ]:
residuals_vs_observation_order_plot_test(model, data)

In [ ]:
r2(model)

## 1.4 Analyse de la variable _transmission_

In [ ]:
# TODO

x = train.transmission
data = DataFrame(y = train.consommation, x = x)
model = create_model(data, [:x])

**Vérification de l'hypothèse de linéarité**

In [ ]:
plot_explanatory_variable(model, data, "Transmission")

**Vérification de l'hypothèse de normalité des erreurs**

In [ ]:
# qqplot_test(model, data)

In [ ]:
shapiro_wilk_test(model, data)

**Vérification de l'hypothèse d'homosédasticité des erreurs**

In [ ]:
residuals_vs_fitted_values_plot_test(model, data)

**Vérification de l'hypothèse d'indépendance des erreurs**

In [ ]:
residuals_vs_observation_order_plot_test(model, data)

In [ ]:
r2(model)

## 1.5 Analyse de la variable _boite_

In [ ]:
data = DataFrame(y = train.consommation, x = train.boite)
model = create_model(data, [:x])

**Vérification de l'hypothèse de linéarité**

In [ ]:
plot_explanatory_variable(model, data, "Cylindrée")

**Vérification de l'hypothèse de normalité des erreurs**

In [ ]:
# qqplot_test(model, data)

In [ ]:
shapiro_wilk_test(model, data)

**Vérification de l'hypothèse d'homosédasticité des erreurs**

In [ ]:
residuals_vs_fitted_values_plot_test(model, data)

**Vérification de l'hypothèse d'indépendance des erreurs**

In [ ]:
residuals_vs_observation_order_plot_test(model, data)

In [ ]:
r2(model)

## 1.6 Analyse de la variable _annee_

In [ ]:
data = DataFrame(y = train.consommation, x = train.annee)
coerce!(data, :x => Multiclass)
model = create_model(data, [:x])

In [ ]:
plot_explanatory_variable(model, data, "Année")

In [ ]:
r2(model)

In [ ]:
# PEut-être
# vif = 1 / (1 - r2(model))

In [ ]:
TODO : RIGDE etc qui fonctonne pas

In [ ]:
TODO : Linear

In [ ]:
# TODO resulat